# Google cluster traces: job-disjoint holdout

[Open in Colab](https://colab.research.google.com/github/crypticsymmetry/improved-giggle/blob/main/notebooks/google_cluster_holdout.ipynb).

Compare placement methods across fixed submission windows and job identities using about 4.5 MB of verified public data. CPU-only; no private dataset, API key or GPU required. Every capacity-grid case is retained, including ties, empty cohorts and uncertified incumbents. The model uses normalized requests and full recorded capacities without reconstructing existing occupancy. This descriptive check within one shard does not prove production scheduler superiority or cross-cluster generalization.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    needs_install = importlib.util.find_spec("assumption_ops.cluster_holdout") is None
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for name in list(sys.modules):
        if name == "assumption_ops" or name.startswith("assumption_ops."):
            del sys.modules[name]
    importlib.invalidate_caches()

import json
import os
from pathlib import Path
from pprint import pprint
from IPython.display import Markdown, display
from assumption_ops.cluster_data import download_cluster
from assumption_ops.cluster_holdout import (
    DEFAULT_WINDOWS, run_cluster_holdout, write_cluster_holdout_csv,
)

## Download verified inputs

Only the complete machine-events object and first task-events shard are fetched. Exact bytes and SHA-256 digests are verified before atomic writes. Set `ASSUMPTION_OPS_CLUSTER_DIR` for offline validation with the same pinned files.

In [ ]:
data_dir = Path(os.environ.get("ASSUMPTION_OPS_CLUSTER_DIR", "cluster_data"))
if os.environ.get("ASSUMPTION_OPS_CLUSTER_DIR"):
    paths = {
        "machines": data_dir / "google_machine_events.csv.gz",
        "tasks": data_dir / "google_task_events.csv.gz",
    }
else:
    paths = download_cluster(data_dir)
assert all(Path(path).is_file() for path in paths.values())
pprint({key: {"path": str(path), "bytes": Path(path).stat().st_size}
        for key, path in paths.items()})

## Declare settings before solving

One development, one validation and six holdout windows use half-open submission intervals. Later windows exclude all jobs admitted earlier, including departed tasks. Settings stay fixed regardless of outcomes. Each snapshot has no previous placement and zero migration cost. Observed terminations remain input rather than consequences of proposals.

In [ ]:
WINDOWS = DEFAULT_WINDOWS
MACHINE_COUNTS = (1, 2, 4, 8)
MAX_TASKS = 64
TIME_LIMIT_SECONDS = 5
experiment_settings = {
    "machine_counts": MACHINE_COUNTS,
    "max_tasks": MAX_TASKS,
    "time_limit_seconds": TIME_LIMIT_SECONDS,
    "windows": [
        {"id": w.id, "split": w.split, "start_us": w.start_us, "end_us": w.end_us}
        for w in WINDOWS
    ],
}
pprint(experiment_settings)

In [ ]:
report = run_cluster_holdout(
    paths["machines"], paths["tasks"], windows=WINDOWS,
    machine_counts=MACHINE_COUNTS, max_tasks=MAX_TASKS,
    time_limit=TIME_LIMIT_SECONDS,
)
pprint(report["protocol"])
print("Cohorts:", len(report["cohorts"]), "capacity cases:", len(report["cases"]))
assert report["controlled_model"] and report["real_source_records"]
assert not report["external_dispatch"]
pprint(report["environment"])

## Check job separation and report every split

Empty cohorts remain visible. Separation includes all admitted jobs, not only tasks still active at evaluation. Summary regret is relative to the MILP proposal: uncertified incumbents must not be called the true optimum. Certification counts and case-level statuses preserve that distinction.

In [ ]:
seen_jobs = set()
cohort_rows = ["| Window | Split | Status | Active tasks | Admitted jobs |",
               "|---|---|---|---:|---:|"]
for cohort in report["cohorts"]:
    window = cohort["window"]
    jobs = set(cohort["metadata"]["selected_job_ids"])
    assert seen_jobs.isdisjoint(jobs), "A job appears in multiple cohorts"
    seen_jobs.update(jobs)
    cohort_rows.append(
        f"| {window['id']} | {window['split']} | {cohort['status']} | "
        f"{cohort['tasks']} | {len(jobs)} |"
    )
display(Markdown("\n".join(cohort_rows)))
rows = ["| Split | Method | Cases | MILP wins | Ties | MILP losses | Mean regret to MILP | Certified MILP cases |",
        "|---|---|---:|---:|---:|---:|---:|---:|"]
for split, summary in report["summary"]["by_split"].items():
    for method, values in summary["methods"].items():
        regret = values["mean_absolute_regret_to_milp"]
        shown_regret = "unavailable" if regret is None else f"{regret:.4g}"
        rows.append(
            f"| {split} | {method} | {values['comparisons']} | "
            f"{values['milp_strict_wins']} | {values['milp_ties']} | "
            f"{values['milp_losses']} | {shown_regret} | "
            f"{values['optimal_milp_comparisons']} |"
        )
display(Markdown("\n".join(rows)))

## Inspect feasibility, solver status and LP bounds

Whole-task assignments satisfy CPU and memory together. The fractional LP is only a lower bound and is never rounded or executed. Trace priority plus one is an assumed admission weight, not money.

In [ ]:
def shown(value):
    return "unavailable" if value is None else f"{value:.4g}"

rows = ["| Window | Machines | Method | Status | Placed | Pending | Score ↓ | LP bound | LP gap |",
        "|---|---:|---|---|---:|---:|---:|---:|---:|"]
for case in report["cases"]:
    bound = case["comparison"]["lp_relaxation"]["lower_bound"]
    for result in case["comparison"]["results"]:
        metrics, costs = result["evaluation"]["metrics"], result["evaluation"]["costs"]
        assert metrics["placed_tasks"] + metrics["pending_tasks"] == metrics["tasks"]
        assert metrics["used_cpu"] <= metrics["cpu_capacity"]
        assert metrics["used_memory"] <= metrics["memory_capacity"]
        assert costs["migration"] == 0
        rows.append(
            f"| {case['window_id']} | {case['machine_count']} | {result['method']} | "
            f"{result['status']} | {metrics['placed_tasks']} | {metrics['pending_tasks']} | "
            f"{costs['total']} | {shown(bound)} | {shown(result['gap_to_lp_bound'])} |"
        )
display(Markdown("\n".join(rows)))

## Export complete evidence

Preserve every case. JSON records the protocol, source hashes, cohort metadata, input fingerprints, proposals and evaluations. CSV rows are alternatives rather than an additive execution ledger.

In [ ]:
output_dir = Path("cluster_holdout_results")
output_dir.mkdir(exist_ok=True)
json_path, csv_path = output_dir / "holdout.json", output_dir / "holdout.csv"
json_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
write_cluster_holdout_csv(report, csv_path)
print("Saved:", json_path.resolve(), csv_path.resolve())
pprint(report["limitations"])
if (importlib.util.find_spec("google") is not None
        and importlib.util.find_spec("google.colab") is not None):
    from google.colab import files
    files.download(str(json_path))
    files.download(str(csv_path))

## Interpretation limits

The windows share one trace and infrastructure; grid cases reuse their window's tasks. Job separation prevents identity reuse but does not establish statistical independence. This experiment performs no fitting, confidence intervals, hypothesis tests or adaptive selection.

Selected capacities do not represent known spare production capacity. Existing occupancy, unsupported affinity, disk, networking, durations, general machine attributes and actual ingestion delay are excluded, and overcommit is not modeled. Results measure feasible proposed scores, not observed production completion, latency or business ROI.

The development region was previously explored. This extension is reproducible, but it is not an untouched external test set or a preregistered study. With no previous plan, stability-aware best fit equals ordinary best fit here; their equality is not independent extra evidence.